# DIM_PERFIL_PACIENTE

Dimensão *junk* com todas as combinações de **sexo** (M, F, I) × **faixa etária**
(00-04, 05-14, 15-29, 30-44, 45-59, 60-74, 75+, IGNORADA). Agrupar esses atributos de
baixa cardinalidade numa dimensão só evita duas FKs a mais na fato.

**Estratégia de carga:** incremental com `MERGE` (SCD tipo 1). Registros novos recebem a surrogate key da sequence; registros existentes só são atualizados se algum atributo mudou. Reexecutar a carga não duplica linhas nem troca as chaves.

**Membro desconhecido:** `SK = -1` ("NAO INFORMADO"), usado pelas fatos quando o código de origem não existe na dimensão.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import mesclar, executar, consultar

In [ ]:
MEMBRO_DESCONHECIDO = """
MERGE INTO DIM_PERFIL_PACIENTE d
USING (SELECT -1 AS SK FROM DUAL) s ON (d.SK_PERFIL = s.SK)
WHEN NOT MATCHED THEN INSERT (SK_PERFIL, SEXO, NOME_SEXO, FAIXA_ETARIA)
VALUES (-1, 'I', 'NAO INFORMADO', 'N/I')
"""
executar(MEMBRO_DESCONHECIDO)

In [ ]:
SQL = """
MERGE INTO DIM_PERFIL_PACIENTE d
USING (
    SELECT sx.SEXO, sx.NOME_SEXO, fx.FAIXA_ETARIA
    FROM (SELECT 'M' AS SEXO, 'MASCULINO' AS NOME_SEXO FROM DUAL
          UNION ALL SELECT 'F', 'FEMININO' FROM DUAL
          UNION ALL SELECT 'I', 'INDETERMINADO' FROM DUAL) sx
    CROSS JOIN (SELECT '00-04' AS FAIXA_ETARIA FROM DUAL
                UNION ALL SELECT '05-14' FROM DUAL UNION ALL SELECT '15-29' FROM DUAL
                UNION ALL SELECT '30-44' FROM DUAL UNION ALL SELECT '45-59' FROM DUAL
                UNION ALL SELECT '60-74' FROM DUAL UNION ALL SELECT '75+' FROM DUAL
                UNION ALL SELECT 'IGNORADA' FROM DUAL) fx
) s
ON (d.SEXO = s.SEXO AND d.FAIXA_ETARIA = s.FAIXA_ETARIA)
WHEN NOT MATCHED THEN INSERT (SK_PERFIL, SEXO, NOME_SEXO, FAIXA_ETARIA)
VALUES (SEQ_DIM_PERFIL.NEXTVAL, s.SEXO, s.NOME_SEXO, s.FAIXA_ETARIA)
"""

In [ ]:
mesclar("DIM_PERFIL_PACIENTE", SQL, origem="(SELECT 1 FROM DUAL CONNECT BY LEVEL <= 24)")

In [ ]:
consultar("SELECT * FROM DIM_PERFIL_PACIENTE ORDER BY 1 FETCH FIRST 10 ROWS ONLY")